In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import hmean
import seaborn as sns
from sklearn.metrics import adjusted_mutual_info_score

# Imputation analysis

In [ ]:
bench1_file = pd.read_csv('../results/cluster_analysis/benchmarking_files/firstbench_2clusters.csv', dtype={'view_combination': str},
                          converters={'y_pred': eval, 'y_pred_idx': eval, 'relative_cluster_sizes': lambda x: eval(x.replace(': ', ':'))})
bench2_file = pd.read_csv('../results/cluster_analysis/benchmarking_files/secondbench.csv', dtype={'view_combination': str},
                          converters={'y_pred': eval, 'y_pred_idx': eval, 'relative_cluster_sizes': lambda x: eval(x.replace(': ', ':'))})
mean_imp_file = pd.read_csv('imputation_mean.csv', dtype={'view_combination': str},
                            converters={'y_pred': eval, 'y_pred_idx': eval, 'relative_cluster_sizes': lambda x: eval(x.replace(': ', ':'))})
median_imp_file = pd.read_csv('imputation_median.csv', dtype={'view_combination': str},
                              converters={'y_pred': eval, 'y_pred_idx': eval, 'relative_cluster_sizes': lambda x: eval(x.replace(': ', ':'))})
knn_imp_file = pd.read_csv('imputation_knn.csv', dtype={'view_combination': str},
                           converters={'y_pred': eval, 'y_pred_idx': eval, 'relative_cluster_sizes': lambda x: eval(x.replace(': ', ':'))})
clinical_data_file = pd.read_csv('../data/TCGA/omics_data/raw/cancer_data_PAAD_clinical_data.tsv', sep='\t', header=0)

In [ ]:
view_combination = "110000"
alg = "NEMO"
bench1_results = bench1_file[(bench1_file['view_combination'] == view_combination) & (bench1_file['algorithm'] == alg)]
bench2_results = bench2_file[bench2_file['algorithm'] == alg]

In [ ]:
noimp_df = pd.concat([bench1_results, bench2_results])
meanimp_df = pd.concat([bench1_results, mean_imp_file])
medianimp_df = pd.concat([bench1_results, median_imp_file])
knnimp_df = pd.concat([bench1_results, knn_imp_file])

In [ ]:
# Edited robustness function (removed dependence on dataset name)
from sklearn.metrics import adjusted_mutual_info_score, normalized_mutual_info_score
def measure_robustness(df):
    results = df.copy(deep=True)
    # Sort labels
    results["sorted_y_pred_idx"] = results["y_pred_idx"].apply(sorted)
    results["sorted_y_pred"] = results.apply(
        lambda row: [row["y_pred"][row["y_pred_idx"].index(patient_id)] for patient_id in row["sorted_y_pred_idx"]],
        axis=1)
    # Divide into complete and missing data dataframes
    complete_df = results[results['missing_percentage'] == 0]
    missing_df = results[results['missing_percentage'] != 0]
    results['robustness'] = np.nan
    # Compare complete vs missing cluster assignments
    for _, base_row in complete_df.iterrows():
        base_idx = base_row["sorted_y_pred_idx"]
        base_pred = base_row["sorted_y_pred"]
        matching_rows = missing_df[
            (missing_df["algorithm"] == base_row["algorithm"]) &
            (missing_df["view_combination"] == base_row["view_combination"]) & 
            (missing_df["n_clusters"] == base_row["n_clusters"]) & 
            (missing_df["run_n"] == base_row["run_n"])]
        for i, miss_row in matching_rows.iterrows():
            miss_idx = miss_row["sorted_y_pred_idx"]
            miss_pred = miss_row["sorted_y_pred"]
            common_patients = list(set(base_idx) & set(miss_idx))
            common_patients_sorted = sorted(common_patients)
            base_filtered = [base_pred[base_idx.index(p)] for p in common_patients_sorted]
            miss_filtered = [miss_pred[miss_idx.index(p)] for p in common_patients_sorted]
            ami = adjusted_mutual_info_score(base_filtered, miss_filtered)
            results.at[i, "robustness"] = ami
        results.at[base_row.name, "robustness"] = 1
    return results

In [ ]:
noimp_robustness = measure_robustness(noimp_df)
noimp_robustness["Imputation mechanism"] = "None"
meanimp_robustness = measure_robustness(meanimp_df)
meanimp_robustness["Imputation mechanism"] = "Mean"
medianimp_robustness = measure_robustness(medianimp_df)
medianimp_robustness["Imputation mechanism"] = "Median"
knnimp_robustness = measure_robustness(knnimp_df)
knnimp_robustness["Imputation mechanism"] = "KNN"

In [ ]:
robustness_prov = pd.concat([noimp_robustness, meanimp_robustness, medianimp_robustness, knnimp_robustness])
hashable_cols = [col for col in robustness_prov.columns if robustness_prov[col].apply(lambda x: not isinstance(x, (list, dict))).all()]
robustness_df = robustness_prov.drop_duplicates(subset=hashable_cols)
robustness_df

In [ ]:
def substitute_ampmech(df):
    amputation_mechanisms = ['edm', 'pm', 'mnar', 'mcar']
    no_amputation_df = df[df['amputation_mechanism'] == 'No']
    new_rows_list = []
    for mechanism in amputation_mechanisms:
        temp_df = no_amputation_df.copy()
        temp_df['amputation_mechanism'] = mechanism
        new_rows_list.append(temp_df)
    new_amputation_rows = pd.concat(new_rows_list, ignore_index=True)
    df_no_no = df[df['amputation_mechanism'] != 'No']
    df_updated = pd.concat([df_no_no, new_amputation_rows], ignore_index=True)
    return df_updated
results_df = substitute_ampmech(robustness_df)

In [ ]:
# Edited function to calculate stability metrics, taking into account imputation
from tqdm import tqdm
import itertools
from sklearn.metrics.cluster import adjusted_mutual_info_score

def calc_ami(df):
    df["sorted_y_pred_idx"] = df["y_pred_idx"].apply(sorted)
    df["sorted_y_pred"] = df.apply(lambda row: [row["y_pred"][row["y_pred_idx"].index(patient_id)] for patient_id in row["sorted_y_pred_idx"]], axis=1)
    df_grouped = df.groupby(["missing_percentage", "amputation_mechanism", "imputation", "view_combination", "algorithm", "Imputation mechanism"], as_index=False).mean(numeric_only=True)
    df_grouped.drop(columns=["run_n", "n_samples"], inplace=True)
    preds_dataset = df[["missing_percentage", "amputation_mechanism", "imputation", "run_n", "sorted_y_pred", "sorted_y_pred_idx", "Imputation mechanism"]]
    for frac in preds_dataset["missing_percentage"].unique():
        df_miss = preds_dataset[preds_dataset["missing_percentage"] == frac]
        for mech in df_miss["amputation_mechanism"].unique():
            df_miss_mech = df_miss[df_miss["amputation_mechanism"] == mech]
            for imp in df_miss_mech["imputation"].unique():
                df_miss_mech_imp = df_miss_mech[df_miss_mech["imputation"] == imp]
                for imech in df_miss_mech_imp["Imputation mechanism"].unique():
                    df_final = df_miss_mech_imp[df_miss_mech_imp["Imputation mechanism"] == imech]
                    amis= []
                    for run_1, run_2 in set(itertools.combinations(df_final["run_n"].unique(), 2)):
                        pred1_alg = df_final.loc[(df_final["run_n"] == run_1), "sorted_y_pred"].to_list()[0]
                        pred2_alg = df_final.loc[(df_final["run_n"] == run_2), "sorted_y_pred"].to_list()[0]
                        pred1_idx = df_final.loc[(df_final["run_n"] == run_1), "sorted_y_pred_idx"].to_list()[0]
                        pred2_idx = df_final.loc[(df_final["run_n"] == run_2), "sorted_y_pred_idx"].to_list()[0]
                        # Only select samples in common for stability metrics
                        common_samples = list(set(pred1_idx) & set(pred2_idx))
                        pred1_common = [pred1_alg[pred1_idx.index(i)] for i in common_samples]
                        pred2_common = [pred2_alg[pred2_idx.index(i)] for i in common_samples]
                        amis.append(adjusted_mutual_info_score(pred1_common, pred2_common))
                    df_grouped.loc[(df_grouped["missing_percentage"] == frac) & (df_grouped["amputation_mechanism"] == mech) & 
                    (df_grouped["imputation"] == imp) & (df_grouped["Imputation mechanism"] == imech), ["AMI"]] = [np.mean(amis)]
    return df_grouped

In [ ]:
def add_normalised_metric(df, variable_to_normalise, metric, greater_is_better=True):
    possible_variables = ["algorithm", "missing_percentage", "amputation_mechanism", "view_combination", "n_clusters", "imputation", "Imputation mechanism"]
    valid_variables = [var for var in possible_variables if var != variable_to_normalise]
    def recursive_loop(subset, remaining_vars, current_filters):
        if not remaining_vars:
            scores = subset[metric].values
            relative_score = scores / scores.max()
            if not greater_is_better:
                relative_score = 1 - relative_score
            condition = True
            for key, value in current_filters.items():
                condition &= (df[key] == value)
            df.loc[condition, f'normalised_{metric}'] = relative_score
            return
        current_var = remaining_vars[0]
        for unique_value in subset[current_var].unique():
            filtered_subset = subset[subset[current_var] == unique_value]
            recursive_loop(filtered_subset, remaining_vars[1:], {**current_filters, current_var: unique_value})
    df[f'normalised_{metric}'] = float('nan')
    recursive_loop(df, valid_variables, {})
    return df

In [ ]:
import warnings
warnings.filterwarnings("ignore")
sil_df = add_normalised_metric(results_df, "missing_percentage", "silhouette", greater_is_better=True)
ami_df = calc_ami(sil_df)
ami_df["AMI"] = ami_df["AMI"].clip(lower=0)
final_df = add_normalised_metric(ami_df, "missing_percentage", "AMI", greater_is_better=True)
final_df['combined_metric'] = final_df[['normalised_silhouette', 'normalised_AMI']].apply(lambda x: hmean(x), axis=1)
final_df.sort_values(["combined_metric"], ascending=False, inplace=True)
final_df

### Pointplots

In [ ]:
from matplotlib import rcParams
sns.set_theme(style='ticks')
rcParams.update({
    'font.size': 11,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica'],
    'axes.labelsize': 11,
    'axes.titlesize': 11,
    'axes.edgecolor': 'black',
    'axes.linewidth': 0.8,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'xtick.direction': 'out',
    'ytick.direction': 'out',
    'xtick.major.size': 3,
    'ytick.major.size': 3,
    'legend.fontsize': 11,
    'legend.frameon': False,
    'savefig.format': 'svg',
    'savefig.dpi': 300,  # Still useful for rasterized elements
    'figure.dpi': 100,
    'figure.figsize': (3.5, 2.5),  # Approx. half-column width
    'figure.constrained_layout.use': True,
    'svg.fonttype': 'none',  # Keep text as editable text (not paths)
    'axes.spines.top': False,
    'axes.spines.right': False,
})

In [ ]:
def plot_pointplots(df, colname_var, colname_metric, ax, ylabel):
    sns.lineplot(x='missing_percentage', y=colname_metric, data=df, hue=colname_var, style=colname_var,
                  palette="colorblind", ax=ax, markers=True, dashes=True, ci=None)
    ax.set_xlabel('Percentage missing data')
    ax.set_ylabel(ylabel)
    ax.set_ylim(bottom=-0.05, top=(max(df[colname_metric])+0.05 if max(df[colname_metric])+0.05 > 1.1  else 1.1))
    plt.tight_layout()
    
fig, ax = plt.subplots(1, 1, figsize=(5, 4))
plot_pointplots(final_df[final_df['missing_percentage'] != 0], 'Imputation mechanism', 'combined_metric', ax, 'General performance score')
ax.legend(title="Imputation mechanism", bbox_to_anchor=(1, 0.8))
ax.set_xticks([20, 40, 60, 80])
# plt.savefig('figures/impmed_gps.svg', bbox_inches='tight')
plt.show()

In [ ]:
def plot_pointplots(df, colname_var, colname_metric, ax, ylabel):
    sns.lineplot(x='missing_percentage', y=colname_metric, data=df, hue=colname_var, style=colname_var,
                  palette="colorblind", ax=ax, markers=True, dashes=True, ci=None)
    ax.set_xlabel('Percentage missing data')
    ax.set_ylabel(ylabel)
    ax.set_ylim(bottom=-0.05, top=(max(df[colname_metric])+0.05 if max(df[colname_metric])+0.05 > 1.1  else 1.1))
    plt.tight_layout()
    
fig, ax = plt.subplots(1, 1, figsize=(5, 4))
plot_pointplots(final_df[final_df['missing_percentage'] != 0], 'Imputation mechanism', 'robustness', ax, 'Robustness')
ax.legend(title="Imputation", bbox_to_anchor=(1.35, 0.8))
ax.set_xticks([20, 40, 60, 80])
# plt.savefig('figures/impmed_robustness.svg', bbox_inches='tight')
plt.show()

In [ ]:
def plot_pointplots(df, colname_var, colname_metric, ax, ylabel):
    sns.lineplot(x='missing_percentage', y=colname_metric, data=df, hue=colname_var, style=colname_var,
                  palette="colorblind", ax=ax, markers=True, dashes=True, ci=None)
    ax.set_xlabel('Percentage missing data')
    ax.set_xticks([20, 40, 60, 80])
    ax.set_ylabel(ylabel)
    ax.set_ylim(bottom=-0.05, top=(max(df[colname_metric])+0.05 if max(df[colname_metric])+0.05 > 1.1  else 1.1))
    plt.tight_layout()

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
plot_pointplots(final_df[final_df['missing_percentage'] != 0], 'Imputation mechanism', 'robustness', ax[0], 'Robustness')
plot_pointplots(final_df[final_df['missing_percentage'] != 0], 'Imputation mechanism', 'combined_metric', ax[1], 'General performance score')
handles_amp, labels_amp = ax[0].get_legend_handles_labels()
for i in ax.flat:
    i.legend().remove()
ax[1].legend(title="Imputation mechanism", bbox_to_anchor=(1, 0.8))
plt.tight_layout(w_pad=5)
plt.savefig('figures/imputationmechs.eps', bbox_inches='tight')
plt.show()

In [ ]:
data_summary = final_df[final_df['missing_percentage'] != 0].groupby(['Imputation mechanism', "missing_percentage"]).agg(
    gps_mean=('combined_metric', 'mean'), 
    gps_std=('combined_metric', 'std'),
    norm_sil_mean=('normalised_silhouette', 'mean'),
    norm_sil_std=('normalised_silhouette', 'std'), 
    norm_AMI_mean=('normalised_AMI', 'mean'),
    norm_AMI_std=('normalised_AMI', 'std'), 
    robustness_mean=("robustness", "mean"), 
    robustness_std=("robustness", "std")).sort_values(by='missing_percentage', ascending=False)
data_summary.to_csv("imputation_summary.csv")